# FIFA 20 Football Player Analysis
## DataMites Internship Project — PRCP-1004-Fifa20

**Beginner-friendly project:** short code cells, clear explanations, simple charts and understandable machine learning.

### Project objectives from the assignment
1. Prepare a complete data analysis report.
2. Explore football skills and group players using clustering.
3. Find the top 10 countries with the most players.
4. Study overall rating against age and discuss when improvement slows.
5. Compare wages of strikers, right-wingers and left-wingers.
6. Compare clustering models and recommend a suitable model.
7. Explain challenges and solutions.

**Dataset:** FIFA 20 Career Mode player data (`players_20.csv`). It contains 18,278 players and 104 columns. This file is small enough to analyze in full.

## 1. Import libraries
Libraries are ready-made tools. Pandas handles tables, Matplotlib makes charts, and Scikit-learn provides clustering algorithms.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

## 2. Load the dataset
Extract `players_20.csv` from the ZIP file and keep it in the same folder as this notebook.

In [ ]:
df = pd.read_csv("players_20.csv")

print("Dataset shape:", df.shape)
df.head()

## 3. Understand the dataset
`info()` shows column names and data types. `describe()` gives basic statistics such as average and maximum.

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Check missing values and duplicates
Missing values are empty cells. Duplicate rows are repeated records.

In [ ]:
print("Missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("Duplicate rows:", df.duplicated().sum())

### Data cleaning
For the three business questions, age, overall, nationality and wage_eur have no missing values in this dataset. The selected 30 football skill attributes used for clustering also have no missing values. We will still check them before clustering.

In [ ]:
print("Missing age:", df["age"].isnull().sum())
print("Missing overall:", df["overall"].isnull().sum())
print("Missing wage:", df["wage_eur"].isnull().sum())

## 5. Task 3.1 — Top 10 countries with the most players
`value_counts()` counts how many players belong to each country. `head(10)` displays the first ten countries.

In [ ]:
top_countries = df["nationality"].value_counts().head(10)
top_countries

In [ ]:
top_countries.sort_values().plot(kind="barh", figsize=(8,5))
plt.title("Top 10 Countries with Most FIFA 20 Players")
plt.xlabel("Number of Players")
plt.ylabel("Country")
plt.show()

### Interpretation
The chart ranks countries by the number of players represented in this FIFA 20 dataset. This measures representation in the game database; it does not necessarily measure the total number of professional footballers produced by each country worldwide.

## 6. Task 3.2 — Overall rating vs age
We calculate the average overall rating for every age. We also show how many players are present at each age so that small age groups are not over-interpreted.

In [ ]:
age_rating = df.groupby("age")["overall"].mean()
age_rating

In [ ]:
age_count = df.groupby("age")["overall"].count()
age_count

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(age_rating.index, age_rating.values, marker="o")
plt.title("Average Overall Rating by Age")
plt.xlabel("Age")
plt.ylabel("Average Overall Rating")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(9,4))
plt.bar(age_count.index, age_count.values)
plt.title("Number of Players at Each Age")
plt.xlabel("Age")
plt.ylabel("Number of Players")
plt.show()

### Interpretation guide
Look for the ages where the average rating rises quickly, then becomes flatter or starts to decline. A flat section means average improvement has slowed. Older ages often have fewer players, so check the player-count chart before making a strong conclusion. This is a cross-sectional comparison of different players, not a follow-up of the same players over time.

## 7. Task 3.3 — Which offensive role earns the highest wage?
The assignment describes strikers as ST, RS and LS, and wingers as RW and LW. We use `team_position` to identify the player's listed club-team role. The wage column is `wage_eur`.

In [ ]:
role_positions = {
    "Striker": ["ST", "RS", "LS"],
    "Right Winger": ["RW"],
    "Left Winger": ["LW"]
}

role_data = df[df["team_position"].isin(
    ["ST", "RS", "LS", "RW", "LW"]
)].copy()

role_data["role"] = role_data["team_position"].replace({
    "ST": "Striker", "RS": "Striker", "LS": "Striker",
    "RW": "Right Winger", "LW": "Left Winger"
})

role_data[["short_name", "team_position", "role", "wage_eur"]].head()

In [ ]:
wage_comparison = role_data.groupby("role")["wage_eur"].agg(
    ["count", "mean", "median"]
)
wage_comparison.sort_values("mean", ascending=False)

In [ ]:
wage_comparison["mean"].sort_values().plot(
    kind="barh", figsize=(8,4)
)
plt.title("Average Wage by Offensive Role")
plt.xlabel("Average Weekly Wage (EUR)")
plt.ylabel("Player Role")
plt.show()

### Interpretation
Compare both mean and median wages. The mean can be influenced by a few very highly paid players, while the median shows the middle wage. State the winning role based on the actual table generated by your notebook. This analysis uses the club team-position field and does not claim to measure every position a player can play.

## 8. Task 2 — Prepare football skills for clustering
Clustering groups players with similar attributes. The assignment says `overall` should be used for discussion but not as a clustering input, because it could dominate the grouping. We therefore leave it out.

We use 30 skill columns covering attacking, technical, movement, power, mentality and defending skills. Position and overall are not used as model inputs.

In [ ]:
skill_columns = [
    "attacking_crossing", "attacking_finishing",
    "attacking_heading_accuracy", "attacking_short_passing",
    "attacking_volleys", "skill_dribbling", "skill_curve",
    "skill_fk_accuracy", "skill_long_passing",
    "skill_ball_control", "movement_acceleration",
    "movement_sprint_speed", "movement_agility",
    "movement_reactions", "movement_balance",
    "power_shot_power", "power_jumping", "power_stamina",
    "power_strength", "power_long_shots",
    "mentality_aggression", "mentality_interceptions",
    "mentality_positioning", "mentality_vision",
    "mentality_penalties", "mentality_composure",
    "defending_marking", "defending_standing_tackle",
    "defending_sliding_tackle", "skill_moves"
]

skills = df[skill_columns]

print("Number of skill columns:", len(skill_columns))
print("Missing skill values:", skills.isnull().sum().sum())

## 9. Scale the skill values
Some measurements can have different ranges. StandardScaler puts the selected skills onto a comparable scale so that features with larger numbers do not dominate simply because of their scale.

In [ ]:
scaler = StandardScaler()
scaled_skills = scaler.fit_transform(skills)

## 10. Build the first clustering model — K-Means
K-Means groups players into a chosen number of clusters. We start with four groups for an easy first analysis. The cluster numbers are labels, not player rankings.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(scaled_skills)

df[["short_name", "overall", "cluster"]].head()

## 11. Understand the player groups
We calculate the average skill values in each group. This helps us describe what makes the clusters different.

In [ ]:
cluster_profile = df.groupby("cluster")[skill_columns].mean()
cluster_profile.round(1)

In [ ]:
cluster_profile[[
    "attacking_finishing", "skill_dribbling",
    "attacking_short_passing", "defending_marking",
    "movement_sprint_speed", "power_strength"
]].plot(kind="bar", figsize=(11,5))
plt.title("Average Selected Skills in Each Player Cluster")
plt.ylabel("Average Skill Rating")
plt.xticks(rotation=0)
plt.show()

### How to describe clusters
Use the cluster profile rather than guessing from the cluster number. For example, a cluster with higher finishing and dribbling may be described as an attack-oriented skill profile. A cluster with stronger tackling and marking may be described as a defence-oriented profile. Names are interpretations of the averages, not official FIFA categories.

## 12. Compare clustering models
We compare K-Means with Agglomerative Clustering using a sample of 3,000 players. A sample keeps this comparison quick on a regular computer. Silhouette score ranges from -1 to 1; a higher score generally means clusters are more clearly separated.

In [ ]:
small_skills = scaled_skills[:3000]

kmeans_test = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_labels = kmeans_test.fit_predict(small_skills)

kmeans_score = silhouette_score(small_skills, kmeans_labels)
print("K-Means Silhouette Score:", kmeans_score)

In [ ]:
agg_model = AgglomerativeClustering(n_clusters=4)
agg_labels = agg_model.fit_predict(small_skills)

agg_score = silhouette_score(small_skills, agg_labels)
print("Agglomerative Silhouette Score:", agg_score)

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["K-Means", "Agglomerative"],
    "Silhouette Score": [kmeans_score, agg_score]
})

model_comparison

### Model recommendation
Compare the silhouette scores. The model with the higher score has better separation on this 3,000-player comparison sample. Also consider running time and ability to handle the full dataset. K-Means is generally easier to scale to all players; Agglomerative Clustering is useful as a comparison but can become slower on larger datasets. Do not choose a winner until you see the actual scores.

## 13. Challenges and solutions
| Challenge | Technique | Reason |
|---|---|---|
| Many columns | Select relevant football skills | Focuses clustering on player abilities |
| Different skill scales | StandardScaler | Makes skill columns comparable |
| Overall rating can dominate | Exclude overall from clustering | Follows the project brief |
| Missing goalkeeper-related data | Do not use goalkeeper-only columns in the selected skill set | Avoids large missing-value problems |
| Choosing number of groups | Start with 4 and compare silhouette | Gives a simple, measurable starting point |
| Clustering labels are abstract | Inspect cluster averages | Helps explain what each group represents |
| Different player roles | Use team_position for wage comparison | Applies the position definitions to a clear role field |

## 14. Conclusion
This project analyzes the FIFA 20 player database, answers the three business questions, and groups players using football skill attributes. The final report should state the top countries, the age trend and the highest-paid offensive role using the outputs generated by your notebook. It should also explain the cluster profiles and compare the clustering models using silhouette score.

The results describe players represented in FIFA 20 Career Mode data. They should not be generalized to every professional footballer worldwide.

## 15. Resume-ready project description
**FIFA 20 Football Player Analysis & Clustering | Python, Pandas, Matplotlib, Scikit-learn**

- Analyzed 18,278 FIFA 20 player records across 104 attributes.
- Performed exploratory analysis of player nationality, age, overall rating and wages.
- Compared wages across striker, right-winger and left-winger roles.
- Applied feature scaling and K-Means clustering to group players using 30 football skill attributes, excluding overall rating.
- Compared K-Means and Agglomerative Clustering using silhouette score and interpreted cluster profiles.

## 16. Final checklist
- [ ] Extract `players_20.csv` from the ZIP and keep it beside the notebook.
- [ ] Run all cells from top to bottom.
- [ ] Check the three business-question outputs and charts.
- [ ] Review cluster profiles before assigning descriptive names.
- [ ] Copy the actual silhouette scores into the report.
- [ ] Save the notebook with outputs for final submission.
- [ ] Practise explaining why scaling and clustering are used.